# 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

# 2. Load Dataset

In [ ]:
data = pd.read_csv("HR_Attrition.csv")
data.sample(10)

In [ ]:
data.shape

In [ ]:
data.columns

# 3. Basic Exploration

In [ ]:
data.info() # statical analysis

In [ ]:
data.describe() # statical analysis

In [ ]:
data["Attrition"].value_counts()

In [ ]:
# Attrition rate:
attrition_rate = (
    data["Attrition"].value_counts(normalize=True)*100
)

print(attrition_rate)

### Only around 16.122449 % of employees left the company while  83.877551% stayed. This indicates an imbalanced dataset. 

In [ ]:
# Numeric col
num_cols = data.select_dtypes(include=np.number)

print(num_cols.shape[1])

In [ ]:
# Categorical columns:
cat_cols = data.select_dtypes(include="object")

print(cat_cols.shape[1])

# 4. Missing Values

In [ ]:
# checking null values in col
data.isnull().sum()

### No missing values were found.

In [ ]:
# checking duplicate values 
data.duplicated().sum()

### No duplicated values were found.

# 5. Drop Useless Columns

In [ ]:
# droping col that not help in tranning
data.drop(
    columns=[
        "EmployeeNumber",
        "Over18",
        "StandardHours",
        "EmployeeCount"
    ],
    inplace=True
)

### EmployeeNumber col means Unique ID , Over18 is a Same value, StandardHours and EmployeeCount also is a same values.

# 6. Convert Target Variable

In [ ]:
data["Attrition"] = data["Attrition"].map({
    "Yes":1,
    "No":0
})

# 7. EDA

In [ ]:
# Attrition by Department
dept_attrition = pd.crosstab(
    data["Department"],
    data["Attrition"],
    normalize="index"
)*100

dept_attrition

In [ ]:
#Visualization
dept_attrition[1].plot(
    kind="bar"
)

plt.ylabel("Attrition Rate %")
plt.savefig("charts/bar plt.png")
plt.show()

### Research & Development has the lowest attrition rate (13.84%), indicating the highest employee retention. In contrast, Sales has the highest attrition rate (20.63%), followed closely by Human Resources (19.05%), suggesting employees leave these departments more frequently.

In [ ]:
# Attrition by Job Role
role_attrition = pd.crosstab(
    data["JobRole"],
    data["Attrition"],
    normalize="index"
)*100
print(role_attrition)
role_attrition[1].sort_values(
    ascending=False
).plot(kind="bar")
plt.savefig("charts/Attrition by Job Role.png")
plt.show()

### Laboratory Technicians (23.94%) and Human Resources employees (23.08%) have the highest attrition rates, indicating they are more likely to leave the company. In contrast, Research Directors (2.50%) and Managers (4.90%) have the lowest attrition rates, showing the strongest employee retention.

In [ ]:
# Monthly Income vs Attrition
sns.boxplot(
    x="Attrition",
    y="MonthlyIncome",
    data=data
)
plt.savefig("charts/Monthly Income vs Attrition.png")
plt.show()

### The box plot compares the salary distribution of employees who stayed versus those who left. It helps determine whether salary is associated with employee attrition.

In [ ]:
# Work Life Balance
sns.countplot(
    x="WorkLifeBalance",
    hue="Attrition",
    data=data
)
plt.savefig("charts/Work Life Balance.png")
plt.show()

### This chart illustrates how employee attrition varies across different work-life balance ratings, indicating whether poor work-life balance contributes to resignations.

In [ ]:
# Years At Company

sns.histplot(
    data=data,
    x="YearsAtCompany",
    hue="Attrition",
    multiple="stack"
)
plt.savefig("charts/Years At Company.png")
plt.show()

### his visualization shows when employees are most likely to leave the organization based on their tenure.

In [ ]:
# converting  categorical columns inot numeric columns
df_encoded = pd.get_dummies(
    data,
    columns=data.select_dtypes(include=["object"]).columns,
    drop_first=True
).astype("int")

In [ ]:
df_encoded.sample(5)

# 9. Split Features & Target

In [ ]:
x = df_encoded.drop(
    "Attrition",
    axis=1
)

y = df_encoded["Attrition"]

# 10. Train Test Split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 11. Scaling

In [ ]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

# 12. Logistic Regression

In [ ]:
lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

lr.fit(x_train, y_train)

In [ ]:
lr_pred = lr.predict(x_test)

lr_prob = lr.predict_proba(x_test)[:,1]

# 13. Random Forest

In [ ]:
rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

rf.fit(x_train,y_train)

rf_pred = rf.predict(x_test)

rf_prob = rf.predict_proba(x_test)[:,1]

# 14. Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(
    random_state=42
)

gb.fit(x_train,y_train)

gb_pred = gb.predict(x_test)

gb_prob = gb.predict_proba(x_test)[:,1]

# 15. Evaluation Function

In [ ]:
def evaluate_model(
    y_true,
    y_pred,
    y_prob,
    model_name
):

    print(model_name)

    print(
        classification_report(
            y_true,
            y_pred
        )
    )

    print(
        "ROC AUC:",
        roc_auc_score(
            y_true,
            y_prob
        )
    )

In [ ]:
evaluate_model(
    y_test,
    lr_pred,
    lr_prob,
    "Logistic Regression"
)

evaluate_model(
    y_test,
    rf_pred,
    rf_prob,
    "Random Forest"
)

evaluate_model(
    y_test,
    gb_pred,
    gb_prob,
    "Gradient Boosting"
)

# 16. Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "ROC_AUC":[
        roc_auc_score(y_test,lr_prob),
        roc_auc_score(y_test,rf_prob),
        roc_auc_score(y_test,gb_prob)
    ]
})

comparison

# 17. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    rf_pred
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d"
)
plt.savefig("charts/Confusion Matrix.png")
plt.show()

### The confusion matrix summarizes the classification performance by showing correct and incorrect predictions for employees who stayed and those who left.

# 18. Feature Importance

In [ ]:
importance = pd.DataFrame({
    "Feature":x.columns,
    "Importance":rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(10)

In [ ]:
# plot 
top10 = importance.head(10)

sns.barplot(
    x="Importance",
    y="Feature",
    data=top10
)
plt.savefig("charts/importance feature.png")
plt.show()

### This chart ranks the most influential features used by the machine learning model to predict employee attrition.

# 19. ROC Curve

In [ ]:
fpr_lr,tpr_lr,_ = roc_curve(
    y_test,
    lr_prob
)

fpr_rf,tpr_rf,_ = roc_curve(
    y_test,
    rf_prob
)

fpr_gb,tpr_gb,_ = roc_curve(
    y_test,
    gb_prob
)

plt.plot(
    fpr_lr,
    tpr_lr,
    label="LR"
)

plt.plot(
    fpr_rf,
    tpr_rf,
    label="RF"
)

plt.plot(
    fpr_gb,
    tpr_gb,
    label="GB"
)

plt.legend()
plt.savefig("charts/ROC Curve.png")
plt.show()

### The ROC curve compares the classification performance of different machine learning models. A curve closer to the top-left corner and a higher AUC score indicate better predictive performance.

# Business Insights (Example Template)
1. Overall attrition rate (balanced or imbalanced dataset)
2. Department with the highest attrition
3. Job role with the highest attrition
4. Years at company where most employees leave
5. Effect of salary, work-life balance, or job satisfaction on attrition

# HR Recommendation Template
HR Recommendation 1:-

HR should focus retention efforts on employees working in the department and job role with the highest attrition by conducting regular one-on-one meetings, career development discussions, and employee satisfaction surveys.

HR Recommendation 2:-

HR should introduce targeted retention programs for employees during their first few years at the company, including mentoring, training opportunities, flexible work policies, and clear career growth plans to reduce early resignations.

# Limitation Template
This model predicts employee attrition using historical HR data, but it cannot capture personal reasons such as family responsibilities, relocation, health issues, or better external job opportunities. Therefore, the predictions should be used as a decision-support tool rather than the sole basis for HR decisions.

## This project developed a machine learning model to predict employee attrition using HR analytics data. After preprocessing, exploratory data analysis, and model comparison, the best-performing model was selected based on evaluation metrics such as Precision, Recall, F1-score, and ROC-AUC. The analysis identified key factors influencing employee turnover, providing actionable insights that HR teams can use to improve employee retention and reduce workforce attrition.